<a href="https://colab.research.google.com/github/fatmasenguler/Mutation_KRAS_analysis/blob/main/Table_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install biopython networkx pandas matplotlib numpy

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 6GOD.pdb to 6GOD.pdb
Saving 6GOF.pdb to 6GOF.pdb


In [ ]:
"""
table2_paper.py  --  PAPER-CONSISTENT Table 2 for 10 channels.
================================================================
Fixes the definition mismatch found against the manuscript.

Manuscript Table 2 caption: "C is computed as the thermal variance of the
channel INTERNAL energy, Var(U)/(kT)^2".  So every column is an
internal-energy quantity:

    C   = Var_P(U)/kT^2 ,   U = kT^2 d(ln W)/dT           (finite differences)
    C_E = Var_P(U_E)/kT^2,  U_E = E_phys                  (physical part)
    C_T = Var_P(U_T)/kT^2,  U_T = kT^2 d(ln tau)/dT       (TOPOLOGICAL INTERNAL energy)
    C_X = 2 Cov_P(U_E,U_T)/kT^2
    ==> C = C_E + C_T + C_X    (verified to <1e-8)

Difference from the earlier 3_cv_decomposition.py: that script used the
topological FREE energy  E_topo = -kT ln(tau)  in place of U_T, which sums
to a DIFFERENT total (Var(E_phys+E_topo)) and does NOT match the paper.
Only C_E was common to both (E_phys is temperature-independent).

WT = 6GOD, G12D = 6GOF.  numpy-only (Colab friendly).
Author: Fatma Ciftci & Burak Erman
"""

import os
import numpy as np

CUTOFF   = 7.8
KT       = 1.0
MIN_LEN  = 2
MAX_LEN  = 9
H        = max(1e-4 * KT, 1e-6)
PDB_FILES = {"WT": "6GOD.pdb", "G12D": "6GOF.pdb"}

CHANNELS = [
    (6,   11,  "Ch1  P-loop (6->11)"),
    (55,  60,  "Ch2  pre-Switch II (55->60)"),
    (110, 117, "Ch3  GBS (110->117)"),
    (141, 146, "Ch4  SAK (141->146)"),
    (19,  142, "Ch5  Lobe linker (19->142)"),
    (12,  35,  "Ch6  G12->Switch I (12->35)"),
    (12,  61,  "Ch7  G12->Q61 SwII (12->61)"),
    (12,  156, "Ch8  G12->alpha5 (12->156)"),
    (12,  170, "Ch9  G12->C-term (12->170)"),
    (35,  61,  "Ch10 Switch I->II (35->61)"),
]


def parse_pdb_ca(filename):
    coords = {}; chosen = None
    with open(filename) as f:
        for line in f:
            if line.startswith("ENDMDL"): break
            if not line.startswith("ATOM"): continue
            if line[12:16].strip() != "CA" or line[16] not in (" ", "A"): continue
            ch = line[21]
            if chosen is None: chosen = ch
            if ch != chosen: continue
            rid = int(line[22:26])
            if rid in coords: continue
            coords[rid] = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return coords


def build_contact_graph(coords, cutoff):
    res = sorted(coords)
    P = np.array([coords[r] for r in res])
    D = np.linalg.norm(P[:, None, :] - P[None, :, :], axis=-1)
    adj = {r: {} for r in res}; dist = {}
    for i, ri in enumerate(res):
        for j in range(i + 1, len(res)):
            if D[i, j] <= cutoff:
                rj = res[j]
                adj[ri][rj] = adj[rj][ri] = D[i, j]
                dist[(ri, rj)] = dist[(rj, ri)] = D[i, j]
    return adj, [r for r in res if adj[r]], dist


def laplacian_pinv(adj, nodes, node_idx, kT):
    n = len(nodes); L = np.zeros((n, n))
    for ri in nodes:
        i = node_idx[ri]
        for rj, d in adj[ri].items():
            if rj in node_idx:
                w = np.exp(-d / kT); L[i, node_idx[rj]] -= w; L[i, i] += w
    return np.linalg.pinv(L)


def enumerate_paths(adj, src, dst, min_len, max_len):
    if src not in adj or dst not in adj: return []
    out = []; stack = [(src, [src], frozenset([src]))]
    while stack:
        node, path, seen = stack.pop()
        if node == dst:
            if len(path) >= min_len: out.append(path)
            continue
        if len(path) >= max_len: continue
        for nb in adj[node]:
            if nb not in seen: stack.append((nb, path + [nb], seen | {nb}))
    return out


def path_logdetY(path, node_idx, K):
    edges = [(path[k], path[k + 1]) for k in range(len(path) - 1)]
    m = len(edges); Y = np.empty((m, m))
    for a, (ia, ja) in enumerate(edges):
        pa, qa = node_idx[ia], node_idx[ja]
        for b, (ib, jb) in enumerate(edges):
            pb, qb = node_idx[ib], node_idx[jb]
            Y[a, b] = K[pa, pb] + K[qa, qb] - K[pa, qb] - K[qa, pb]
    s, ld = np.linalg.slogdet(Y)
    return ld if (s > 0 and np.isfinite(ld)) else None


def channel_decomposition(paths, dist, node_idx, Km, Kp, Kmn, kT, h):
    UE, UT, lnW = [], [], []
    for p in paths:
        ep = sum(dist[(p[k], p[k + 1])] for k in range(len(p) - 1))
        lt_m = path_logdetY(p, node_idx, Km)
        lt_p = path_logdetY(p, node_idx, Kp)
        lt_n = path_logdetY(p, node_idx, Kmn)
        if lt_m is None or lt_p is None or lt_n is None: continue
        UE.append(ep)                              # U_E = E_phys (physical internal energy)
        UT.append(kT * kT * (lt_p - lt_n) / (2 * h))  # U_T = kT^2 d ln(tau)/dT
        lnW.append(-ep / kT + lt_m)
    if not lnW: return None
    UE = np.array(UE); UT = np.array(UT); lnW = np.array(lnW)
    w = np.exp(lnW - lnW.max()); P = w / w.sum(); kT2 = kT * kT
    mE = P @ UE; mT = P @ UT
    C_E = (P @ (UE - mE) ** 2) / kT2
    C_T = (P @ (UT - mT) ** 2) / kT2
    C_X = 2 * (P @ ((UE - mE) * (UT - mT))) / kT2
    return {"n": len(lnW), "C_E": C_E, "C_T": C_T, "C_X": C_X, "C": C_E + C_T + C_X}


def run_structure(fname):
    coords = parse_pdb_ca(fname)
    adj, nodes, dist = build_contact_graph(coords, CUTOFF)
    ni = {r: i for i, r in enumerate(nodes)}
    Km = laplacian_pinv(adj, nodes, ni, KT)
    Kp = laplacian_pinv(adj, nodes, ni, KT + H)
    Kmn = laplacian_pinv(adj, nodes, ni, KT - H)
    out = {}
    for src, dst, lbl in CHANNELS:
        if src not in ni or dst not in ni:
            out[lbl] = "endpoint missing"; continue
        paths = enumerate_paths(adj, src, dst, MIN_LEN, MAX_LEN)
        if not paths:
            out[lbl] = f"no path L in [{MIN_LEN}..{MAX_LEN}]"; continue
        r = channel_decomposition(paths, dist, ni, Km, Kp, Kmn, KT, H)
        out[lbl] = r if r is not None else "no valid path"
    return out, len(nodes)


def main():
    try:
        from google.colab import files as cf; in_colab = True
    except ImportError:
        in_colab = False
    for _, fn in PDB_FILES.items():
        if not os.path.exists(fn):
            if in_colab: print(f"Upload {fn}:"); cf.upload()
            else: raise FileNotFoundError(fn)

    print("=" * 92)
    print(f"  Table 2 (paper-consistent)  |  C = Var(U)/kT^2  |  cutoff={CUTOFF} A  "
          f"L=[{MIN_LEN}..{MAX_LEN}]  h={H:.1e}")
    print("=" * 92)
    R = {}
    for st, fn in PDB_FILES.items():
        R[st], nn = run_structure(fn); print(f"  {st:<5} ({fn}): {nn} residues")

    print(f"\n{'Channel':<26}{'C_E':>10}{'C_T':>10}{'C_X':>10}{'C':>10}{'n':>9}")
    for st in ("WT", "G12D"):
        print(f"----- {st}")
        for _, _, lbl in CHANNELS:
            r = R[st][lbl]
            if not isinstance(r, dict): print(f"{lbl:<26}  [{r}]"); continue
            print(f"{lbl:<26}{r['C_E']:>10.4f}{r['C_T']:>10.4f}{r['C_X']:>10.4f}{r['C']:>10.4f}{r['n']:>9}")

    print(f"\n{'Channel':<26}{'C_WT':>9}{'C_G12D':>9}{'dC':>9}{'dC_E':>9}{'dC_T':>9}{'dC_X':>9}"
          f"{'nWT':>9}{'nG12D':>10}")
    with open("table2_paper.csv", "w") as f:
        f.write("Channel,C_WT,C_G12D,dC,dC_E,dC_T,dC_X,n_WT,n_G12D\n")
        for _, _, lbl in CHANNELS:
            w, m = R["WT"][lbl], R["G12D"][lbl]
            if not (isinstance(w, dict) and isinstance(m, dict)):
                print(f"{lbl:<26}  [{w if not isinstance(w,dict) else m}]"); continue
            dC = m["C"] - w["C"]; dE = m["C_E"] - w["C_E"]; dT = m["C_T"] - w["C_T"]; dX = m["C_X"] - w["C_X"]
            print(f"{lbl:<26}{w['C']:>9.2f}{m['C']:>9.2f}{dC:>+9.2f}{dE:>+9.2f}{dT:>+9.2f}{dX:>+9.2f}"
                  f"{w['n']:>9}{m['n']:>10}")
            f.write(f"{lbl},{w['C']:.4f},{m['C']:.4f},{dC:.4f},{dE:.4f},{dT:.4f},{dX:.4f},{w['n']},{m['n']}\n")
    print("\nSaved: table2_paper.csv")


if __name__ == "__main__":
    main()


  Table 2 (paper-consistent)  |  C = Var(U)/kT^2  |  cutoff=7.8 A  L=[2..9]  h=1.0e-04
  WT    (6GOD.pdb): 172 residues
  G12D  (6GOF.pdb): 172 residues

Channel                          C_E       C_T       C_X         C        n
----- WT
Ch1  P-loop (6->11)          49.0282   39.8980  -82.2203    6.7058   946282
Ch2  pre-Switch II (55->60)   41.9096   36.8377  -72.5434    6.2040   638606
Ch3  GBS (110->117)          23.3624   16.4048  -31.7553    8.0118   824774
Ch4  SAK (141->146)          47.8882   40.4487  -82.2414    6.0955   934827
Ch5  Lobe linker (19->142)   46.3465   41.8459  -82.7332    5.4593   864439
Ch6  G12->Switch I (12->35)   59.6538   57.2346 -112.9176    3.9708   321499
Ch7  G12->Q61 SwII (12->61)   38.4605   32.4598  -68.8646    2.0557   124222
Ch8  G12->alpha5 (12->156)   21.4475   19.2844  -36.1609    4.5710   350220
Ch9  G12->C-term (12->170)    5.5031    2.4989   -4.2036    3.7985    12372
Ch10 Switch I->II (35->61)   51.2550   48.4205  -96.4691    3.2064   18787